Date: 11/06/2024 <br>
Desc: Tests BN with CPT computed from using dataset details CSV file <br>
      Uses horizontal distance, height, MCS, and USI as inputs

In [1]:
import pandas as pd
import numpy as np 
import math
import os
from tqdm import tqdm
from sklearn.metrics import accuracy_score

def get_mcs_index(df_in):
    '''
    Gets the MCS index based on modulation and bitrate column of the df_in
    '''
    df = df_in.copy()
    df["MCS"] = ''
    df.loc[(df["Modulation"] == "BPSK") & (df["Bitrate"] == 6.5), "MCS"] = 0 # MCS Index 0
    df.loc[(df["Modulation"] == "QPSK") & (df["Bitrate"] == 13), "MCS"] = 1 # MCS Index 0
    df.loc[(df["Modulation"] == "QPSK") & (df["Bitrate"] == 19.5), "MCS"] = 2 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM16") & (df["Bitrate"] == 26), "MCS"] = 3 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM16") & (df["Bitrate"] == 39), "MCS"] = 4 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 52), "MCS"] = 5 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 58.5), "MCS"] = 6 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 65), "MCS"] = 7 # MCS Index 0

    return df

def find_nearest_value(value, array):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]

def find_nearest_index(value, array):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return idx


In [2]:
# Set paths to datasets etc.
CPT_PATH = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_ParrotAR2/bn_cpts/parrotar2_reliability_bn_CPT_Uplink.csv"
HDIST_BIN_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/bn_ckpt/djispark_reliability_bn_hdist_bins_Downlink.npy"
HEIGHT_BIN = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/bn_ckpt/djispark_reliability_bn_height_bins_Downlink.npy"
TEST_DATA_PATH = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_ParrotAR2/test_dataset_1_processed/Uplink_Reliability.csv"
MAX_HDIST = 700 # To accomodate for the change in largest horizontal distance
# Set minimum probability for failure mode to be considered
MIN_FAILURE_PROB = 0.01
RELIABILITY_TH = [0.99, 0.999] # For calculation max AE in reliable region

# Load CPT and bin files
cpt_df = pd.read_csv(CPT_PATH)
# hdist_bin = np.load(HDIST_BIN_PATH)
# height_bin = np.load(HEIGHT_BIN)
hdist_bin = np.arange(0, 710, 10) # For associating each hdist to its nearest value in train dataset
height_bin = np.arange(60, 330, 30) # For associating each height to its nearest value in train dataset

# Load testing dataset and process
test_data_df = pd.read_csv(TEST_DATA_PATH)
test_data_df = test_data_df.loc[test_data_df["Horizontal_Distance"] <= MAX_HDIST]
test_data_df = get_mcs_index(test_data_df)
    # If associating each horizontal distance and height with the closest values in training dataset:
test_data_df["Horizontal_Distance_Class"] = pd.cut(test_data_df["Horizontal_Distance"], bins=hdist_bin, right=False, include_lowest=True, labels=np.arange(0, len(hdist_bin)-1))
# test_data_df["Horizontal_Distance_Class"] = test_data_df["Horizontal_Distance"].apply(find_nearest_index, args=([hdist_bin]))
test_data_df["Height_Class"] = pd.cut(test_data_df["Height"], bins=height_bin, right=False, include_lowest=True, labels=np.arange(0, len(height_bin)-1))
# test_data_df["Height_Class"] = test_data_df["Height"].apply(find_nearest_index, args=([height_bin]))
test_data_df["UAV_Sending_Interval_Class"] = test_data_df["UAV_Sending_Interval"].replace({10:0, 20:1, 66.7:2, 100:3}) # Change sending interval categorial to numeric
test_data_df["Reliability"] = (test_data_df["Num_Reliable"] / test_data_df["Num_Sent"]).values
test_data_df["Delay_Excd_Prob"] = (test_data_df["Num_Delay_Excd"] / test_data_df["Num_Sent"]).values
test_data_df["Queue_Overflow_Prob"] = (test_data_df["Num_Q_Overflow"] / test_data_df["Num_Sent"]).values
test_data_df["Incr_Rcvd_Prob"] = (test_data_df["Num_Incr_Rcvd"] / test_data_df["Num_Sent"]).values

predicted_reliability = [] # To store reliability predictions
predicted_incr_rcvd = [] # To store incorrectly received probability predictions
predicted_delay_excd = [] # To store delay exceeded probability predictions
predicted_q_ovflw = [] # To store queue overflow probability predictions
for row in tqdm(test_data_df.itertuples()):
    # predictions = cpt_df.loc[(row.Horizontal_Distance_Class,row.Height_Class,row.UAV_Sending_Interval_Class,row.MCS)]
    predictions = cpt_df.loc[(cpt_df["Horizontal_Distance_Class"]==row.Horizontal_Distance_Class) & (cpt_df["Height_Class"]==row.Height_Class) &
                             (cpt_df["UAV_Sending_Interval_Class"]==row.UAV_Sending_Interval_Class) & (cpt_df["MCS"]==row.MCS)]
    try:
        predicted_reliability.append(predictions["Reliability"].values[0])
        predicted_delay_excd.append(predictions["Prob_Delay_Excd"].values[0])
        predicted_q_ovflw.append(predictions["Prob_Queue_Overflow"].values[0])
        predicted_incr_rcvd.append(predictions["Prob_Incr_Rcvd"].values[0])
    except:
        print(row)
        print(predictions)
        break

test_data_df['Predicted_Reliability'] = predicted_reliability
test_data_df['Predicted_Delay_Excd_Prob'] = predicted_delay_excd
test_data_df['Predicted_Queue_Overflow_Prob'] = predicted_q_ovflw
test_data_df['Predicted_Incr_Rcvd_Prob'] = predicted_incr_rcvd

test_data_df["Reliability_Class"] = pd.cut(test_data_df["Reliability"], bins=[-0.1,0.1,0.5,0.9,1], labels=["Low", "ModeratelyLow", "ModeratelyHigh", "High"])
test_data_df["Predicted_Reliability_Class"] = pd.cut(test_data_df["Predicted_Reliability"], bins=[-0.1,0.1,0.5,0.9,1], labels=["Low", "ModeratelyLow", "ModeratelyHigh", "High"])
test_data_df["Failure_Mode"] = test_data_df[["Queue_Overflow_Prob", "Incr_Rcvd_Prob", "Delay_Excd_Prob"]].idxmax(axis=1)
test_data_df["Predicted_Failure_Mode"] = test_data_df[["Predicted_Queue_Overflow_Prob", "Predicted_Incr_Rcvd_Prob", "Predicted_Delay_Excd_Prob"]].idxmax(axis=1)
# Replace label for Failure Mode with "None" if none of the failure modes have a probability > 5%
test_data_df.loc[(test_data_df["Queue_Overflow_Prob"] < MIN_FAILURE_PROB) & (test_data_df["Incr_Rcvd_Prob"] < MIN_FAILURE_PROB) & (test_data_df["Delay_Excd_Prob"] < MIN_FAILURE_PROB),["Failure_Mode"]] = "None"
test_data_df.loc[(test_data_df["Predicted_Queue_Overflow_Prob"] < MIN_FAILURE_PROB) & (test_data_df["Predicted_Incr_Rcvd_Prob"] < MIN_FAILURE_PROB) & (test_data_df["Predicted_Delay_Excd_Prob"] < MIN_FAILURE_PROB),["Predicted_Failure_Mode"]] = "None"

# Compute the model accuracy and mean abs err
failure_mode = test_data_df["Failure_Mode"].replace({"Queue_Overflow_Prob":1, "Incr_Rcvd_Prob":2, "Delay_Excd_Prob":3, "None":4})
failure_mode_predicted = test_data_df["Predicted_Failure_Mode"].replace({"Predicted_Queue_Overflow_Prob":1, "Predicted_Incr_Rcvd_Prob":2, "Predicted_Delay_Excd_Prob":3, "None":4})
reliability_accuracy = accuracy_score(test_data_df["Reliability_Class"], test_data_df["Predicted_Reliability_Class"])
failure_mode_accuracy = accuracy_score(failure_mode, failure_mode_predicted)
reliability_mae = np.mean(abs(test_data_df['Reliability'].values - test_data_df['Predicted_Reliability'].values))
queue_overflow_mae = np.mean(abs(test_data_df['Queue_Overflow_Prob'].values - test_data_df['Predicted_Queue_Overflow_Prob'].values))
incr_rcvd_mae = np.mean(abs(test_data_df['Incr_Rcvd_Prob'].values - test_data_df['Predicted_Incr_Rcvd_Prob'].values))
delay_excd_mae = np.mean(abs(test_data_df['Delay_Excd_Prob'].values - test_data_df['Predicted_Delay_Excd_Prob'].values))
reliability_maxae = np.max(abs(test_data_df['Reliability'].values - test_data_df['Predicted_Reliability'].values))
queue_overflow_maxae = np.max(abs(test_data_df['Queue_Overflow_Prob'].values - test_data_df['Predicted_Queue_Overflow_Prob'].values))
incr_rcvd_maxae = np.max(abs(test_data_df['Incr_Rcvd_Prob'].values - test_data_df['Predicted_Incr_Rcvd_Prob'].values))
delay_excd_maxae = np.max(abs(test_data_df['Delay_Excd_Prob'].values - test_data_df['Predicted_Delay_Excd_Prob'].values))

# Print results
print("Reliability - Accuracy: {}, MAE: {}, MaxAE: {}".format(reliability_accuracy, reliability_mae, reliability_maxae))
print("Failure Mode - Accuracy: {}".format(failure_mode_accuracy))
print("Queue Overflow - MeanAE: {}, MaxAE: {}".format(queue_overflow_mae, queue_overflow_maxae))
print("Incorrectly Received - MeanAE: {}, MaxAE: {}".format(incr_rcvd_mae, incr_rcvd_maxae))
print("Delay Exceeded - MeanAE: {}, MaxAE: {}".format(delay_excd_mae, delay_excd_maxae))
print("Average Failure Mode Mean AE: {}".format(np.mean([queue_overflow_mae, incr_rcvd_mae, delay_excd_mae])))

for reliability_th in RELIABILITY_TH:
    # Get the Max Abs Err of reliability, but only when either the simulated/predicted reliability is above the threshold
    # test_data_reliable_df = test_data_df.loc[(test_data_df["Reliability"]>=reliability_th) | (test_data_df["Predicted_Reliability"]>=reliability_th)]
    # rel_err = test_data_reliable_df['Reliability'].values - test_data_reliable_df['Predicted_Reliability'].values
    # reliability_maxae_reliable = np.max(abs(rel_err))
    # reliability_mean_reliable = np.mean(abs(rel_err))
    # print("Reliability - MeanAE_Reliable_Region: {}, MaxAE_Reliable_Region: {}".format(reliability_mean_reliable, reliability_maxae_reliable))

    # UNCOMMENT TO EVALUATE reliability_state_accuracy over reliable/predicted_reliable region only
    # Get the accuracy of predicting reliability above the threshold
    # test_data_df["Reliable_State"] = test_data_df["Reliability"] >= reliability_th
    # test_data_df["Predicted_Reliable_State"] = test_data_df["Predicted_Reliability"] >= reliability_th
    # reliability_state_accuracy = accuracy_score(test_data_df["Reliable_State"], test_data_df["Predicted_Reliable_State"])

    # # UNCOMMENT TO EVALUATE reliability_state_accuracy over entire region
    # # Get the accuracy of predicting reliability above the threshold
    test_data_df["Reliable_State"] = test_data_df["Reliability"] >= reliability_th
    test_data_df["Predicted_Reliable_State"] = test_data_df["Predicted_Reliability"] >= reliability_th
    reliability_state_accuracy = accuracy_score(test_data_df["Reliable_State"], test_data_df["Predicted_Reliable_State"])
    print("Reliability - Accuracy_Reliable_Region >= {}: {}".format(reliability_th, reliability_state_accuracy))

# Save results to file
# test_data_df.to_csv("/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJIMavicAir/Test_Dataset_2_NP10000_DJIMavicAir_Downlink_Reliability_RESULTS_bn_pandas_cpt.csv")

307it [00:00, 1532.28it/s]

2240it [00:01, 1580.34it/s]

Reliability - Accuracy: 0.9821428571428571, MAE: 0.010339069196339286, MaxAE: 0.9921099999999999
Failure Mode - Accuracy: 0.9745535714285715
Queue Overflow - MeanAE: 0.01338544402308209, MaxAE: 0.88945
Incorrectly Received - MeanAE: 0.00040918750133927227, MaxAE: 0.006740000000000001
Delay Exceeded - MeanAE: 0.016886548951742623, MaxAE: 0.94518
Average Failure Mode Mean AE: 0.010227060158721329
Reliability - Accuracy_Reliable_Region >= 0.99: 0.9883928571428572
Reliability - Accuracy_Reliable_Region >= 0.999: 0.9928571428571429
